In [1]:
from pyspark.sql import SparkSession

In [2]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, LongType, DateType, TimeType,FloatType

In [3]:
from pyspark.sql.functions import col

In [5]:
spark = SparkSession.builder \
    .appName("MiApp") \
    .getOrCreate()    

C:\Users\hever\anaconda3\Lib\site-packages\pyspark\testing\utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


In [8]:
schema1 = StructType([
    StructField('order_item_id',
                StringType(), True),
    StructField('order_id',
                StringType(), True),
    StructField('product_id',
                StringType(), True),
    StructField('quantity',
                IntegerType(), True),
    StructField('unit_price',
                FloatType(), True),
      StructField('discount_amount',
                FloatType(), True),
     StructField('line_total',
                FloatType(), True),
])

In [15]:
schema2 = StructType([
    StructField('product_id',
                StringType(), True),
    StructField('sku',
                StringType(), True),
    StructField('product_name',
                StringType(), True),
    StructField('category',
                StringType(), True),
    StructField('brand',
                StringType(), True),
      StructField('price',
                FloatType(), True),
     StructField('cost',
                FloatType(), True),
     StructField('stock',
                IntegerType(), True),
     StructField('created_at',
                DateType(), True),
])

In [21]:
order_items = spark.read.csv("../data/batch/order_items.csv", schema=schema1,  header=True)

In [16]:
products = spark.read.csv("../data/batch/products.csv",schema=schema2,   header=True)

In [22]:
order_items.show()

+-------------+-----------+----------+--------+----------+---------------+----------+
|order_item_id|   order_id|product_id|quantity|unit_price|discount_amount|line_total|
+-------------+-----------+----------+--------+----------+---------------+----------+
| OI00000001-1|ORD00000001| PROD00267|       2|   16380.3|            0.0|   32760.6|
| OI00000001-2|ORD00000001| PROD00326|       1|  13890.31|            0.0|  13890.31|
| OI00000001-3|ORD00000001| PROD00209|       3|   1876.23|            0.0|   5628.69|
| OI00000001-4|ORD00000001| PROD00325|       1|  24143.74|            0.0|  24143.74|
| OI00000002-1|ORD00000002| PROD00284|       4|  10726.51|            0.0|  42906.04|
| OI00000002-2|ORD00000002| PROD00218|       2|   9795.93|         979.59|  18612.27|
| OI00000002-3|ORD00000002| PROD00255|       1|   4895.99|          244.8|   4651.19|
| OI00000003-1|ORD00000003| PROD00126|       1|  12025.16|        1202.52|  10822.64|
| OI00000003-2|ORD00000003| PROD00230|       2|   8334

In [17]:
products.show()

+----------+-------------+--------------------+-----------+------+--------+--------+-----+----------+
|product_id|          sku|        product_name|   category| brand|   price|    cost|stock|created_at|
+----------+-------------+--------------------+-----------+------+--------+--------+-----+----------+
| PROD00001|SKU-MOD-00001| Producto Moda 00001|       Moda| Nexum|23270.16|15599.06|  268|2025-03-07|
| PROD00002|SKU-BEL-00002|Producto Belleza ...|    Belleza|  Nova|11732.07| 9004.63|  244|2025-08-22|
| PROD00003|SKU-MOD-00003| Producto Moda 00003|       Moda| Orion|24605.45|18848.17|  430|2025-07-13|
| PROD00004|SKU-JUG-00004|Producto Juguetes...|   Juguetes|  Nova| 7422.27| 4771.39|  469|2025-02-05|
| PROD00005|SKU-HOG-00005|Producto Hogar 00005|      Hogar| Zenda|12790.76| 6178.83|  338|2026-03-18|
| PROD00006|SKU-BEL-00006|Producto Belleza ...|    Belleza| Nexum| 8918.67| 4268.65|  257|2025-01-13|
| PROD00007|SKU-HOG-00007|Producto Hogar 00007|      Hogar| Zenda|18525.88| 9500.8

In [32]:
productos_vendidos = products.alias("a").join(order_items.alias("b"), col("a.product_id") == col("b.product_id"), "inner")

In [38]:
productos_vendidos2 = productos_vendidos.select("a.product_id", "order_item_id", "sku", "product_name", "price", "cost", "stock", "quantity", "unit_price", "line_total")

In [39]:
productos_vendidos2.show()

+----------+-------------+-------------+--------------------+--------+--------+-----+--------+----------+----------+
|product_id|order_item_id|          sku|        product_name|   price|    cost|stock|quantity|unit_price|line_total|
+----------+-------------+-------------+--------------------+--------+--------+-----+--------+----------+----------+
| PROD00267| OI00000001-1|SKU-JUG-00267|Producto Juguetes...| 16380.3| 9841.81|   90|       2|   16380.3|   32760.6|
| PROD00326| OI00000001-2|SKU-ELE-00326|Producto Electrón...|13890.31| 7778.62|   20|       1|  13890.31|  13890.31|
| PROD00209| OI00000001-3|SKU-ELE-00209|Producto Electrón...| 1876.23| 1287.97|  236|       3|   1876.23|   5628.69|
| PROD00325| OI00000001-4|SKU-DEP-00325|Producto Deportes...|24143.74|12391.35|  306|       1|  24143.74|  24143.74|
| PROD00284| OI00000002-1|SKU-AUT-00284|Producto Autos 00284|10726.51| 8531.92|  438|       4|  10726.51|  42906.04|
| PROD00218| OI00000002-2|SKU-HOG-00218|Producto Hogar 00218| 97

In [40]:
productos_vendidos2.write.mode("overwrite").parquet("../data/batch/salida_ventas.parquet")

Py4JJavaError: An error occurred while calling o170.parquet.
: java.lang.RuntimeException: java.io.FileNotFoundException: java.io.FileNotFoundException: HADOOP_HOME and hadoop.home.dir are unset. -see https://cwiki.apache.org/confluence/display/HADOOP2/WindowsProblems
	at org.apache.hadoop.util.Shell.getWinUtilsPath(Shell.java:790)
	at org.apache.hadoop.util.Shell.getSetPermissionCommand(Shell.java:299)
	at org.apache.hadoop.util.Shell.getSetPermissionCommand(Shell.java:315)
	at org.apache.hadoop.fs.RawLocalFileSystem.setPermission(RawLocalFileSystem.java:1179)
	at org.apache.hadoop.fs.RawLocalFileSystem.mkOneDirWithMode(RawLocalFileSystem.java:861)
	at org.apache.hadoop.fs.RawLocalFileSystem.mkdirsWithOptionalPermission(RawLocalFileSystem.java:901)
	at org.apache.hadoop.fs.RawLocalFileSystem.mkdirs(RawLocalFileSystem.java:873)
	at org.apache.hadoop.fs.RawLocalFileSystem.mkdirsWithOptionalPermission(RawLocalFileSystem.java:900)
	at org.apache.hadoop.fs.RawLocalFileSystem.mkdir